In [1]:
import sys
sys.path.append("..")

from src.solvers.equations_base import BaseEquationSolver

In [2]:
import re
from typing import List, Tuple, Dict, Optional

import pandas as pd
import statistics
from tqdm import tqdm

tqdm.pandas()


In [3]:
data = pd.read_csv("../data/raw/train.csv")

In [4]:
data["prompt_eda"] = data.prompt.str.split('.').apply(lambda x: x[0])

In [5]:
task_classes = {
    "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers":  "bit manipulation",
    "In Alice's Wonderland, secret encryption rules are used on text": "encryption",
    "In Alice's Wonderland, numbers are secretly converted into a different numeral system": "conversion to diff numeral system",
    "In Alice's Wonderland, a secret unit conversion is applied to measurements": "unit conversion",
    "In Alice's Wonderland, the gravitational constant has been secretly changed": "gravitational",
    "In Alice's Wonderland, a secret set of transformation rules is applied to equations": "equations transformation"
}

In [6]:
data["label"] = data.prompt_eda.map(task_classes)
data["label"].value_counts()

label
bit manipulation                     1602
gravitational                        1597
unit conversion                      1594
encryption                           1576
conversion to diff numeral system    1576
equations transformation             1555
Name: count, dtype: int64

In [7]:
eq_df = data[data['label'] == 'equations transformation'].copy()#.sample(50, random_state=1244124)

In [8]:
solver = BaseEquationSolver()

# 1. Функция для извлечения и классификации
def classify_prompt(prompt: str) -> str:
    examples, target = solver._extract_sections(prompt)
    if not examples:
        return "Extraction Failed"
    return solver._classify_task(examples, target)[0]

# Применяем классификацию ко всем промптам
eq_df['task_type'] = eq_df.prompt.apply(classify_prompt)

# 2. Вывод количества задач по каждому классу
print("=== Статистика по классам задач ===")
class_counts = eq_df['task_type'].value_counts()
print(class_counts.to_string())
print("\n")

# 3. Функция для решения и сравнения с эталонным ответом
def evaluate_solver(row):
    # Вызываем солвер
    result = solver.solve(row['prompt']) 
    
    # Строгая проверка типа: если это словарь (новая версия)
    if isinstance(result, dict):
        predicted = result.get('answer')
        debug_log = result.get('debug', [])
    # Если это строка или None (старая версия)
    else:
        predicted = result
        debug_log = ["Дебаг недоступен: используется старая версия солвера."]
    
    # Приводим оба ответа к строке и удаляем лишние пробелы для корректного сравнения
    true_answer = str(row['answer']).strip()
    
    # Обрабатываем None
    pred_answer = str(predicted).strip() if predicted is not None else "nan"
    
    is_correct = (true_answer == pred_answer)
    
    # Возвращаем 3 колонки
    return pd.Series(
        [pred_answer, is_correct, debug_log], 
        index=['predicted_answer', 'is_correct', 'debug_log']
    )

=== Статистика по классам задач ===
task_type
Cryptarithm (CSP)              823
AST Brute-force                660
Pseudo-Math (Format/String)     72




In [9]:
samples_per_class = 3

# Список уникальных классов в датафрейме
task_types = eq_df['task_type'].unique()

for task_type in task_types:
    print(f"\n{'='*5} Класс: {task_type} {'='*5}")
    
    # Фильтруем задачи текущего класса
    subset = eq_df[eq_df['task_type'] == task_type]
    
    # Берем первые N примеров (можно заменить .head() на .sample(), чтобы брать случайные)
    sample_df = subset.sample(samples_per_class)
    
    for idx, row in sample_df.iterrows():
        print(f"\nID: {row.get('id', idx)}")
        print(row['prompt'].strip())
        print(f"Ответ: {row['answer']}")
        print("-" * 5)


===== Класс: Cryptarithm (CSP) =====

ID: 685be3a7
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
|?*|& = [$?
`&*<| = `>`$
`<*`| = $|&!
Now, determine the result for: !?+`:
Ответ: >$$
-----

ID: 524cb5c6
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
$?:>` = $?>`
\<:\` = \<\`
\{@)? = @$)
{`:?> = {`?>
Now, determine the result for: $>:>\
Ответ: $>>\
-----

ID: 8fdf7ccf
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
%$*)] = <>
!]+)) = +$"
"<*>> = )#]
""?<% = #$]$
Now, determine the result for: !<?$#
Ответ: <$$%
-----

===== Класс: AST Brute-force =====

ID: 9250a0a8
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
63?39 = 103
50?12 = 63
37#85 = 3145
21@77 = 14
88?10 = 99
Now, determine the result for: 57@18
Ответ: 3
-----

ID: 6ab04968
I

In [10]:
solver.solve("""In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
>:^@% = $]%@
@@}]# = :/
&]}]% = ]#
@$[$# = [%<
Now, determine the result for: $>}@/""")

{'answer': None,
 'debug': ['Error: Too many unique characters (11 > 10) for a base-10 numerical system. Problem is mathematically ill-posed.']}

In [11]:
from pandarallel import pandarallel

pandarallel.initialize(nb_workers=24, progress_bar=True)


INFO: Pandarallel will run on 24 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [12]:
print("Запуск солвера. Пожалуйста, подождите...")
results_df = eq_df.parallel_apply(evaluate_solver, axis=1)
eq_df = pd.concat([eq_df, results_df], axis=1)

# 5. Вывод процента решенных задач (accuracy) по каждому классу
print("\n=== Результаты решения по классам ===")
# Группируем по типу задачи и считаем среднее значение (True = 1, False = 0)
accuracy_stats = eq_df.groupby('task_type')['is_correct'].agg(['count', 'sum', 'mean'])

for task_type, row in accuracy_stats.iterrows():
    total = int(row['count'])
    correct = int(row['sum'])
    accuracy_percent = row['mean'] * 100
    
    print(f"Класс: {task_type}")
    print(f"Решено: {correct} из {total} ({accuracy_percent:.2f}%)\n")

# Итоговая точность по всему датасету
total_tasks = len(eq_df)
total_correct = eq_df['is_correct'].sum()
overall_accuracy = (total_correct / total_tasks) * 100 if total_tasks > 0 else 0

print(f"=== Общий итог ===")
print(f"Всего решено: {total_correct} из {total_tasks} ({overall_accuracy:.2f}%)")

Запуск солвера. Пожалуйста, подождите...



=== Результаты решения по классам ===
Класс: AST Brute-force
Решено: 511 из 660 (77.42%)

Класс: Cryptarithm (CSP)
Решено: 142 из 823 (17.25%)

Класс: Pseudo-Math (Format/String)
Решено: 72 из 72 (100.00%)

=== Общий итог ===
Всего решено: 725 из 1555 (46.62%)


In [13]:
failed_pseudo_math = eq_df[
    (eq_df['task_type'] == 'Cryptarithm (CSP)') & 
    (eq_df['is_correct'] == False)
]

print(f"Всего нерешенных задач Cryptarithm (CSP): {len(failed_pseudo_math)}\n")

# Берем первые 10 примеров для анализа
sample_to_analyze = failed_pseudo_math.sample(20, random_state=13)

for idx, row in sample_to_analyze.iterrows():
    print("="*60)
    print(f"ID: {row.get('id', idx)}")
    print(f"--- Prompt ---\n{row['prompt']}")
    print(f"--- Answers ---")
    print(f"True Answer: {row['answer']}")
    print(f"Predicted:   {row['predicted_answer']}")
    print(f"DEBUG: {row['debug_log']}")
print("="*60)

Всего нерешенных задач Cryptarithm (CSP): 681

ID: 48194d70
--- Prompt ---
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
[:*"' = [@[
:\/{^ = ^[|@
^[/@{ = :^[[
[\/'[ = '[^
Now, determine the result for: ^@/[:
--- Answers ---
True Answer: ^[#|
Predicted:   nan
DEBUG: ['After exhaustive logical propagation, no valid mathematical mapping could be found. This indicates an ill-posed cryptarithm.']
ID: c7844441
--- Prompt ---
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
^%|:> = |%<
<:|/: = |&`
/&|<: = @$
&>}%% = $@
:`}>$ = %<$
Now, determine the result for: :/[$^
--- Answers ---
True Answer: :/$^
Predicted:   nan
DEBUG: ['Error: Too many unique characters (11 > 10) for a base-10 numerical system. Problem is mathematically ill-posed.']
ID: 303c84ea
--- Prompt ---
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few ex

In [14]:
failed_pseudo_math = eq_df[
    (eq_df['task_type'] == 'Cryptarithm (CSP)') & 
    (eq_df['is_correct'] == True)
]

print(f"Всего решенных задач Cryptarithm (CSP): {len(failed_pseudo_math)}\n")

# Берем первые 10 примеров для анализа
sample_to_analyze = failed_pseudo_math.sample(40, random_state=312)

for idx, row in sample_to_analyze.iterrows():
    if ("Concat L+R" not in str(row['debug_log'])) and ("Concat R+L" not in str(row['debug_log'])):
        print("="*60)
        print(f"ID: {row.get('id', idx)}")
        print(f"--- Prompt ---\n{row['prompt']}")
        print(f"--- Answers ---")
        print(f"True Answer: {row['answer']}")
        print(f"Predicted:   {row['predicted_answer']}")
        print(f"DEBUG: {row['debug_log']}")
print("="*60)

Всего решенных задач Cryptarithm (CSP): 142

ID: 5f4b89b7
--- Prompt ---
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
'>-&$ = $$
(\-(> = /
(|*'# = |(''
>>*>/ = '\'
Now, determine the result for: ']->$
--- Answers ---
True Answer: >\
Predicted:   >\
DEBUG: ['Step 1: Hypothesis Generation (Structural vs Mathematical)', 'First, we check if the operators represent simple string manipulations.', 'No consistent structural patterns were found. Therefore, the symbols must conceal digits (0-9) representing a cryptarithm.', '\nStep 2: Cryptarithm Decoding & Deductive Reasoning', 'We must decode 10 unique symbols into distinct digits (0-9). To do this logically, we apply mathematical heuristics:', '\nHeuristic 1: Magnitude Bounds', 'By analyzing the length of the operands versus the length of the result (e.g., carrying over in addition/multiplication), we can restrict the possible mathematical operations.', " -> For operator '-', t

In [15]:
class_counts = eq_df['task_type']
class_counts

8       Cryptarithm (CSP)
26      Cryptarithm (CSP)
29        AST Brute-force
39      Cryptarithm (CSP)
41      Cryptarithm (CSP)
              ...        
9459      AST Brute-force
9466    Cryptarithm (CSP)
9467    Cryptarithm (CSP)
9482      AST Brute-force
9484    Cryptarithm (CSP)
Name: task_type, Length: 1555, dtype: str

In [16]:
analyze = eq_df[eq_df['task_type'] == "Cryptarithm (CSP)"]
analyze

,id,prompt,answer,prompt_eda,label,task_type,predicted_answer,is_correct,debug_log
8,00457d26,"In Alice's Wonderland, a secret set of transfo...",@&,"In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),@&,True,[Step 1: Hypothesis Generation (Structural vs ...
26,00c032a8,"In Alice's Wonderland, a secret set of transfo...",\^?,"In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),nan,False,[Timeout: Exceeded 10.0s limit. Complex branch...
39,012cab1f,"In Alice's Wonderland, a secret set of transfo...",|@{,"In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),nan,False,[Timeout: Exceeded 10.0s limit. Complex branch...
41,0133bcec,"In Alice's Wonderland, a secret set of transfo...",\([#,"In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),\([#,True,[Step 1: Hypothesis Generation (Structural vs ...
52,017a871e,"In Alice's Wonderland, a secret set of transfo...",\:,"In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),nan,False,[Timeout: Exceeded 10.0s limit. Complex branch...
...,...,...,...,...,...,...,...,...,...
9415,fdbdf50c,"In Alice's Wonderland, a secret set of transfo...","-\""","In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),nan,False,[Timeout: Exceeded 10.0s limit. Complex branch...
9434,fe6da79d,"In Alice's Wonderland, a secret set of transfo...",:&]!,"In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),nan,False,[Error: Too many unique characters (11 > 10) f...
9466,ff0e37ae,"In Alice's Wonderland, a secret set of transfo...","%'""`","In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),"%'""`",True,[Step 1: Hypothesis Generation (Structural vs ...
9467,ff121f08,"In Alice's Wonderland, a secret set of transfo...",#<,"In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),nan,False,[Error: Too many unique characters (11 > 10) f...


In [17]:
def analyze_prompt(prompt: str) -> str:
    examples, target = solver._extract_sections(prompt)
    results = []
    for line in examples.strip().split('\n'):
        if '=' in line:
            lhs_raw, rhs_raw = line.split('=', 1)
            results.append(lhs_raw[2])
    return results


def analyze_prompt_all(prompt: str) -> str:
    examples, target = solver._extract_sections(prompt)
    results = []
    for line in examples.strip().split('\n'):
        if '=' in line:
            lhs_raw, rhs_raw = line.split('=', 1)
            results.append([i for i in lhs_raw] + [i for i in rhs_raw])
    return results

In [18]:
results = analyze.prompt.apply(analyze_prompt_all)

In [19]:
set([v for j in results.to_list() for x in j for v in x])

{' ',
 '!',
 '"',
 '#',
 '$',
 '%',
 '&',
 "'",
 '(',
 ')',
 '*',
 '+',
 '-',
 '/',
 ':',
 '<',
 '>',
 '?',
 '@',
 '[',
 '\\',
 ']',
 '^',
 '`',
 '{',
 '|',
 '}'}

In [20]:
len(set([v for j in results.to_list() for x in j for v in x]))

27

In [21]:

for v in set([v for j in results.to_list() for x in j for v in x]):
    print(f"{v} = {ord(v)}")

- = 45
] = 93
# = 35
& = 38
% = 37
[ = 91
$ = 36
" = 34
@ = 64
! = 33
^ = 94
< = 60
} = 125
) = 41
: = 58
` = 96
  = 32
( = 40
\ = 92
+ = 43
> = 62
* = 42
| = 124
/ = 47
{ = 123
' = 39
? = 63


In [22]:
result = list(set([v for j in results.to_list() for x in j for v in x]))

In [23]:
result.sort()

In [24]:
result

[' ',
 '!',
 '"',
 '#',
 '$',
 '%',
 '&',
 "'",
 '(',
 ')',
 '*',
 '+',
 '-',
 '/',
 ':',
 '<',
 '>',
 '?',
 '@',
 '[',
 '\\',
 ']',
 '^',
 '`',
 '{',
 '|',
 '}']

In [25]:
d = {}
for i, r in enumerate(result[1:]):
    d[r] = i+1
    

In [26]:
print(d)

{'!': 1, '"': 2, '#': 3, '$': 4, '%': 5, '&': 6, "'": 7, '(': 8, ')': 9, '*': 10, '+': 11, '-': 12, '/': 13, ':': 14, '<': 15, '>': 16, '?': 17, '@': 18, '[': 19, '\\': 20, ']': 21, '^': 22, '`': 23, '{': 24, '|': 25, '}': 26}


In [27]:
)'-\^ = ^$
:$+// = @^
()+$\ = ^'
\!+/( = /'
Now, determine the result for: ))*!(
--- Answers ---
True Answer: @@//

SyntaxError: unmatched ')' (3185388340.py, line 1)

In [ ]:
86-1921 = 213
133+1212 = 1721
78+319 = 215
190+127 = 76

78*07

должно получится 17171212

In [ ]:
97-2022 = 226
144+1313 = 1822
78+420 = 217
201+138 = 137

теперь посчтитать для 99*18
должно получится 18181313

In [ ]:
нет... это не так
а что если?
[9,7]- [19, 21] = [2, 13]

SyntaxError: invalid syntax (2759583327.py, line 1)